# bd2.ipynb

`cf2` 폴더 안의 모든 단순 탐지 JSON과 `sample` 폴더 안의 모든 영상을 대상으로 싸움/스토킹 후보를 추출하는 노트북입니다.

핵심 흐름:

1. `cf2/*.json` 전체 로드
2. 각 JSON의 `frames -> objects`에서 person/human 객체 추출
3. `track_id`가 없는 cf2 결과에 대해 bbox 기반 pseudo-tracking 생성
4. 싸움 후보 탐지
   - 사람 bbox에서 손/발 후보 영역을 근사
   - 손/발 영역이 상대 bbox와 겹치거나 가까워지는 상황이 짧은 구간에서 2회 이상 반복되면 fight 후보
5. 스토킹 후보 탐지
   - 한 사람이 다른 사람 뒤쪽에서 일정 거리 유지
   - 두 사람이 같은 방향으로 일정 시간 이동하면 stalking 후보
6. 영상이 있으면 이벤트 구간 클립 추출
7. 결과를 JSON으로 저장


In [1]:
# ==============================
# Cell 1. 기본 import
# ==============================
import os
import json
import math
import cv2
import numpy as np

from pathlib import Path
from collections import defaultdict
from itertools import combinations
from tqdm.auto import tqdm


In [2]:
# ==============================
# Cell 2. 경로 설정
# ==============================
# 네 환경에서 보통 아래 둘 중 하나일 가능성이 큼.
# - 노트북이 프로젝트 루트에 있으면 ./cf2, ./sample
# - Windows 사용자 폴더 바로 아래에 있으면 C:/Users/우성한/cf2, C:/Users/우성한/sample

BASE_DIR = Path.cwd()

# 직접 지정하고 싶으면 아래 두 줄만 바꾸면 됨.
CF2_DIR = Path(r"C:\Users\우성한\cf2")
SAMPLE_DIR = Path(r"C:\Users\우성한\sample")

# 위 경로가 없으면 현재 작업 폴더 기준으로 자동 fallback
if not CF2_DIR.exists():
    CF2_DIR = BASE_DIR / "cf2"

if not SAMPLE_DIR.exists():
    SAMPLE_DIR = BASE_DIR / "sample"

OUTPUT_DIR = BASE_DIR / "outputs" / "prediction" / "bd2"
EVENT_CLIP_DIR = OUTPUT_DIR / "event_clips"
EVENT_FRAME_DIR = OUTPUT_DIR / "event_frames"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVENT_CLIP_DIR.mkdir(parents=True, exist_ok=True)
EVENT_FRAME_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR   :", BASE_DIR)
print("CF2_DIR    :", CF2_DIR, "exists =", CF2_DIR.exists())
print("SAMPLE_DIR :", SAMPLE_DIR, "exists =", SAMPLE_DIR.exists())
print("OUTPUT_DIR :", OUTPUT_DIR)


BASE_DIR   : C:\Users\우성한
CF2_DIR    : C:\Users\우성한\cf2 exists = True
SAMPLE_DIR : C:\Users\우성한\sample exists = False
OUTPUT_DIR : C:\Users\우성한\outputs\prediction\bd2


In [3]:
# ==============================
# Cell 3. 파라미터
# ==============================

# tracking
PERSON_CLASSES = {"person", "human"}
MIN_TRACK_LENGTH = 6
MAX_MISSING_FRAMES = 8
TRACK_IOU_TH = 0.12
TRACK_CENTER_DIST_TH = 180

# fight
FIGHT_WINDOW_SEC = 3.0
FIGHT_MIN_CONTACT_COUNT = 2
FIGHT_MIN_ACTIVE_SPEED = 6.0
FIGHT_LIMB_NEAR_TH = 45
FIGHT_MERGE_GAP_SEC = 2.0

# stalking
STALK_WINDOW_SEC = 5.0
STALK_MIN_RATIO = 0.55
STALK_MIN_DIST = 50
STALK_MAX_DIST = 420
STALK_DIR_SIM_TH = 0.55
STALK_BEHIND_DOT_TH = 10
STALK_DISTANCE_STD_TH = 130
STALK_MERGE_GAP_SEC = 3.0

# clip extraction
EXTRACT_EVENT_CLIPS = True
CLIP_MARGIN_SEC = 1.0

# cf2 json에 fps가 없으면 sample 영상 fps를 쓰고, 영상도 없으면 이 값을 사용
DEFAULT_FPS = 30.0

VIDEO_EXTS = [".mp4", ".avi", ".mov", ".mkv", ".wmv"]


In [4]:
# ==============================
# Cell 4. 파일 목록 확인
# ==============================
json_files = sorted(CF2_DIR.glob("*.json")) if CF2_DIR.exists() else []

video_files = []
if SAMPLE_DIR.exists():
    for ext in VIDEO_EXTS:
        video_files.extend(SAMPLE_DIR.rglob(f"*{ext}"))
video_files = sorted(video_files)

print("json 개수 :", len(json_files))
for p in json_files[:10]:
    print(" JSON :", p.name)

print("\nvideo 개수:", len(video_files))
for p in video_files[:10]:
    print(" VIDEO:", p.name)


json 개수 : 10
 JSON : bs_e4387_c1.json
 JSON : bs_e4387_c2.json
 JSON : es_e3768_c1.json
 JSON : es_e3768_c2.json
 JSON : md_e2122_c1.json
 JSON : md_e2122_c2.json
 JSON : mp_e2730_c1.json
 JSON : mp_e2730_c2.json
 JSON : ms_e3500_c1.json
 JSON : ms_e3500_c2.json

video 개수: 0


In [5]:
# ==============================
# Cell 5. 기본 유틸
# ==============================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def safe_float(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return default

def safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default

def bbox_xyxy(bbox):
    if bbox is None or len(bbox) < 4:
        return None
    x1, y1, x2, y2 = map(float, bbox[:4])
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1, y1, x2, y2]

def bbox_center(b):
    x1, y1, x2, y2 = b
    return [(x1 + x2) / 2, (y1 + y2) / 2]

def foot_point(b):
    x1, y1, x2, y2 = b
    return [(x1 + x2) / 2, y2]

def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    if union <= 0:
        return 0.0
    return inter / union

def bbox_intersects(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    return max(ax1, bx1) < min(ax2, bx2) and max(ay1, by1) < min(ay2, by2)

def point_dist(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def bbox_min_dist(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    if bbox_intersects(a, b):
        return 0.0
    dx = max(bx1 - ax2, ax1 - bx2, 0)
    dy = max(by1 - ay2, ay1 - by2, 0)
    return math.hypot(dx, dy)

def vec(p1, p2):
    return [p2[0] - p1[0], p2[1] - p1[1]]

def norm(v):
    return math.hypot(v[0], v[1])

def cosine(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 <= 1e-6 or n2 <= 1e-6:
        return 0.0
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)

def mean(xs):
    return sum(xs) / len(xs) if xs else 0.0

def std(xs):
    if len(xs) < 2:
        return 0.0
    m = mean(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))


In [6]:
# ==============================
# Cell 6. JSON/영상 매칭
# ==============================
def video_stem_candidates(video_name):
    if not video_name:
        return []
    p = Path(str(video_name))
    return [p.stem, str(video_name), p.name]

def build_video_index(video_files):
    index = {}
    for vp in video_files:
        index[vp.stem] = vp
        index[vp.name] = vp
    return index

VIDEO_INDEX = build_video_index(video_files)

def find_matching_video(json_path, raw):
    candidates = []
    if isinstance(raw, dict):
        candidates.extend(video_stem_candidates(raw.get("video_name")))
        candidates.extend(video_stem_candidates(raw.get("source_video")))
        candidates.extend(video_stem_candidates(raw.get("file_name")))
    candidates.append(Path(json_path).stem)

    for cand in candidates:
        if not cand:
            continue
        cand_path = Path(str(cand))
        if cand_path.exists():
            return cand_path
        if cand in VIDEO_INDEX:
            return VIDEO_INDEX[cand]
        stem = cand_path.stem
        if stem in VIDEO_INDEX:
            return VIDEO_INDEX[stem]

    json_stem = Path(json_path).stem
    for key, vp in VIDEO_INDEX.items():
        if json_stem == Path(key).stem or json_stem in Path(key).stem or Path(key).stem in json_stem:
            return vp
    return None

def get_video_info(video_path):
    if video_path is None:
        return {
            "exists": False,
            "fps": DEFAULT_FPS,
            "width": None,
            "height": None,
            "frame_count": None,
            "duration_sec": None
        }
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "exists": False,
            "fps": DEFAULT_FPS,
            "width": None,
            "height": None,
            "frame_count": None,
            "duration_sec": None
        }
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-6:
        fps = DEFAULT_FPS
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {
        "exists": True,
        "fps": float(fps),
        "width": width,
        "height": height,
        "frame_count": frame_count,
        "duration_sec": round(frame_count / fps, 3) if fps else None
    }


In [7]:
# ==============================
# Cell 7. cf2 JSON 정규화
# ==============================
def extract_class_name(obj):
    for k in ["class", "class_name", "label", "name"]:
        if k in obj:
            return str(obj[k]).lower()
    return ""

def extract_bbox(obj):
    for k in ["bbox", "box", "xyxy"]:
        if k in obj:
            return bbox_xyxy(obj[k])
    if all(k in obj for k in ["x1", "y1", "x2", "y2"]):
        return bbox_xyxy([obj["x1"], obj["y1"], obj["x2"], obj["y2"]])
    return None

def normalize_cf2_json(raw, json_path, fps=DEFAULT_FPS):
    # cf2 dict 형식과 tracking_result list 형식을 둘 다 일부 호환
    detections_by_frame = defaultdict(list)

    if isinstance(raw, dict) and "frames" in raw:
        video_name = raw.get("video_name", Path(json_path).stem)

        for fr in raw.get("frames", []):
            frame_id = safe_int(fr.get("frame_id", fr.get("frame", 0)))
            time_sec = fr.get("time_sec")
            if time_sec is None:
                time_sec = frame_id / fps

            for obj_idx, obj in enumerate(fr.get("objects", [])):
                cls = extract_class_name(obj)
                if cls not in PERSON_CLASSES:
                    continue

                b = extract_bbox(obj)
                if b is None:
                    continue

                det = {
                    "video_name": video_name,
                    "frame_id": frame_id,
                    "time_sec": float(time_sec),
                    "class_name": cls,
                    "confidence": safe_float(obj.get("confidence", obj.get("score", 0.0))),
                    "bbox": b,
                    "source_obj_index": obj_idx,
                    "source_track_id": obj.get("track_id")
                }
                detections_by_frame[frame_id].append(det)

        return video_name, detections_by_frame

    elif isinstance(raw, list):
        video_name = Path(json_path).stem
        for item in raw:
            cls = str(item.get("class_name", item.get("class", ""))).lower()
            if cls not in PERSON_CLASSES:
                continue

            b = extract_bbox(item)
            if b is None:
                continue

            frame_id = safe_int(item.get("frame_id", item.get("frame", 0)))
            time_sec = item.get("time_sec")
            if time_sec is None:
                time_sec = frame_id / fps

            if item.get("video_name"):
                video_name = item.get("video_name")

            detections_by_frame[frame_id].append({
                "video_name": video_name,
                "frame_id": frame_id,
                "time_sec": float(time_sec),
                "class_name": cls,
                "confidence": safe_float(item.get("confidence", item.get("score", 0.0))),
                "bbox": b,
                "source_obj_index": None,
                "source_track_id": item.get("track_id")
            })

        return video_name, detections_by_frame

    else:
        return Path(json_path).stem, detections_by_frame


In [8]:
# ==============================
# Cell 8. pseudo-tracking
# ==============================
def detection_center(det):
    return bbox_center(det["bbox"])

def track_match_score(prev_det, cur_det):
    i = iou(prev_det["bbox"], cur_det["bbox"])
    d = point_dist(detection_center(prev_det), detection_center(cur_det))
    dist_score = max(0.0, 1.0 - d / TRACK_CENTER_DIST_TH)
    return i * 2.0 + dist_score

def enrich_point(det, tid):
    b = det["bbox"]
    return {
        "track_id": str(tid),
        "frame_id": int(det["frame_id"]),
        "time_sec": float(det["time_sec"]),
        "bbox": [round(x, 2) for x in b],
        "center": [round(x, 2) for x in bbox_center(b)],
        "foot_point": [round(x, 2) for x in foot_point(b)],
        "confidence": det.get("confidence")
    }

def clean_tracks(tracks):
    cleaned = {}
    for tid, pts in tracks.items():
        pts = sorted(pts, key=lambda x: x["frame_id"])
        if len(pts) >= MIN_TRACK_LENGTH:
            cleaned[str(tid)] = pts
    return cleaned

def build_tracks_from_detections(detections_by_frame):
    # source_track_id가 대부분 있는 경우 그대로 사용
    all_dets = [d for dets in detections_by_frame.values() for d in dets]
    with_tid = [d for d in all_dets if d.get("source_track_id") is not None]

    if len(all_dets) > 0 and len(with_tid) / len(all_dets) >= 0.7:
        tracks = defaultdict(list)
        for d in all_dets:
            if d.get("source_track_id") is None:
                continue
            tid = str(d["source_track_id"])
            tracks[tid].append(enrich_point(d, tid))
        return clean_tracks(tracks)

    next_tid = 1
    active = {}
    tracks = defaultdict(list)

    for frame_id in sorted(detections_by_frame.keys()):
        cur_dets = detections_by_frame[frame_id]

        active = {
            tid: st for tid, st in active.items()
            if frame_id - st["last_frame_id"] <= MAX_MISSING_FRAMES
        }

        assigned_det = set()
        assigned_track = set()

        candidates = []
        for tid, st in active.items():
            prev_det = st["last_det"]
            for idx, cur_det in enumerate(cur_dets):
                d = point_dist(detection_center(prev_det), detection_center(cur_det))
                i = iou(prev_det["bbox"], cur_det["bbox"])
                if i >= TRACK_IOU_TH or d <= TRACK_CENTER_DIST_TH:
                    score = track_match_score(prev_det, cur_det)
                    candidates.append((score, tid, idx))

        candidates.sort(reverse=True, key=lambda x: x[0])

        for score, tid, idx in candidates:
            if tid in assigned_track or idx in assigned_det:
                continue

            cur_det = cur_dets[idx]
            assigned_track.add(tid)
            assigned_det.add(idx)

            p = enrich_point(cur_det, tid)
            tracks[tid].append(p)
            active[tid] = {
                "last_frame_id": frame_id,
                "last_det": cur_det
            }

        for idx, cur_det in enumerate(cur_dets):
            if idx in assigned_det:
                continue

            tid = str(next_tid)
            next_tid += 1

            p = enrich_point(cur_det, tid)
            tracks[tid].append(p)
            active[tid] = {
                "last_frame_id": frame_id,
                "last_det": cur_det
            }

    return clean_tracks(tracks)

def build_frame_map(tracks):
    fm = defaultdict(dict)
    for tid, pts in tracks.items():
        for p in pts:
            fm[p["frame_id"]][str(tid)] = p
    return fm

def track_point_map(points):
    return {p["frame_id"]: p for p in points}

def get_prev_point(tmap, frame_id):
    prevs = [fid for fid in tmap.keys() if fid < frame_id]
    if not prevs:
        return None
    return tmap[max(prevs)]

def compute_speed(tmap, frame_id):
    cur = tmap.get(frame_id)
    prev = get_prev_point(tmap, frame_id)
    if cur is None or prev is None:
        return 0.0
    return point_dist(cur["center"], prev["center"])

def compute_direction(tmap, frame_id):
    cur = tmap.get(frame_id)
    prev = get_prev_point(tmap, frame_id)
    if cur is None or prev is None:
        return [0.0, 0.0]
    return vec(prev["center"], cur["center"])

def summarize_tracks(tracks):
    out = {}
    for tid, pts in tracks.items():
        out[tid] = {
            "track_length": len(pts),
            "first_frame": pts[0]["frame_id"],
            "last_frame": pts[-1]["frame_id"],
            "first_time": round(pts[0]["time_sec"], 3),
            "last_time": round(pts[-1]["time_sec"], 3),
            "first_center": pts[0]["center"],
            "last_center": pts[-1]["center"]
        }
    return out


In [9]:
# ==============================
# Cell 9. 손/발 후보 bbox 근사
# ==============================
def person_limb_boxes(person_bbox):
    # pose/keypoint 없이 person bbox만 있으므로 손/발 후보 영역을 bbox 내부 비율로 근사
    x1, y1, x2, y2 = person_bbox
    w = max(1.0, x2 - x1)
    h = max(1.0, y2 - y1)

    hand_w = 0.24 * w
    hand_h = 0.22 * h
    foot_w = 0.26 * w
    foot_h = 0.18 * h

    lh = [
        x1 - 0.08 * w,
        y1 + 0.32 * h,
        x1 + hand_w,
        y1 + 0.32 * h + hand_h
    ]
    rh = [
        x2 - hand_w,
        y1 + 0.32 * h,
        x2 + 0.08 * w,
        y1 + 0.32 * h + hand_h
    ]
    lf = [
        x1 + 0.18 * w,
        y2 - foot_h,
        x1 + 0.18 * w + foot_w,
        y2 + 0.06 * h
    ]
    rf = [
        x2 - 0.18 * w - foot_w,
        y2 - foot_h,
        x2 - 0.18 * w,
        y2 + 0.06 * h
    ]

    return {
        "left_hand": lh,
        "right_hand": rh,
        "left_foot": lf,
        "right_foot": rf
    }

def limb_contact_signal(a, b):
    # a의 손/발 후보 영역이 b의 person bbox와 겹치거나 충분히 가까우면 접촉 후보.
    # b -> a도 같이 확인.
    candidates = []

    for owner, p, other in [("a", a, b), ("b", b, a)]:
        limb_boxes = person_limb_boxes(p["bbox"])
        for limb_name, limb_box in limb_boxes.items():
            md = bbox_min_dist(limb_box, other["bbox"])
            overlap = bbox_intersects(limb_box, other["bbox"])
            if overlap or md <= FIGHT_LIMB_NEAR_TH:
                candidates.append({
                    "owner": owner,
                    "limb": limb_name,
                    "min_distance": round(md, 2),
                    "overlap": bool(overlap),
                    "limb_bbox": [round(x, 2) for x in limb_box]
                })

    return len(candidates) > 0, candidates[:6]


In [10]:
# ==============================
# Cell 10. 이벤트 병합/최종판정
# ==============================
def event_pair_key(ev):
    if ev.get("event_class") == "fight":
        return tuple(sorted(ev.get("track_pair", [])))

    if ev.get("event_class") == "stalking":
        return (
            str(ev.get("follower_track_id")),
            str(ev.get("target_track_id"))
        )

    return None

def merge_events(events, event_class, gap_sec):
    if not events:
        return []

    events = sorted(events, key=lambda x: (x["start_time"], x["end_time"]))
    merged = []

    for ev in events:
        if not merged:
            merged.append(ev)
            continue

        last = merged[-1]
        same_class = ev.get("event_class") == event_class and last.get("event_class") == event_class
        same_pair = event_pair_key(ev) == event_pair_key(last)
        close = ev["start_time"] <= last["end_time"] + gap_sec

        if same_class and same_pair and close:
            last["end_frame"] = max(last["end_frame"], ev["end_frame"])
            last["end_time"] = round(max(last["end_time"], ev["end_time"]), 3)
            last["confidence"] = max(last["confidence"], ev["confidence"])
        else:
            merged.append(ev)

    return merged

def decide_final(fight_events, stalking_events):
    predictions = sorted(
        fight_events + stalking_events,
        key=lambda x: x.get("confidence", 0),
        reverse=True
    )

    if not predictions:
        return {
            "detected": False,
            "main_event": "none",
            "confidence": 0.0,
            "reason": "싸움 또는 스토킹 조건을 만족하는 사람 track 관계를 찾지 못함.",
            "predictions": []
        }

    main = predictions[0]
    return {
        "detected": True,
        "main_event": main["event_class"],
        "confidence": main["confidence"],
        "predictions": predictions
    }


In [11]:
# ==============================
# Cell 11. 싸움 탐지
# ==============================
def detect_fight(tracks, frame_map):
    if len(tracks) < 2:
        return []

    tmaps = {tid: track_point_map(pts) for tid, pts in tracks.items()}
    pair_signals = defaultdict(list)

    for frame_id in sorted(frame_map.keys()):
        persons = frame_map[frame_id]
        if len(persons) < 2:
            continue

        for tid_a, tid_b in combinations(sorted(persons.keys()), 2):
            a, b = persons[tid_a], persons[tid_b]

            contact, contacts = limb_contact_signal(a, b)
            speed_a = compute_speed(tmaps[tid_a], frame_id)
            speed_b = compute_speed(tmaps[tid_b], frame_id)

            active = max(speed_a, speed_b) >= FIGHT_MIN_ACTIVE_SPEED
            center_dist = point_dist(a["center"], b["center"])

            signal = contact and active

            pair_key = tuple(sorted([tid_a, tid_b]))
            pair_signals[pair_key].append({
                "frame_id": int(frame_id),
                "time_sec": round(float(a["time_sec"]), 3),
                "signal": bool(signal),
                "contact": bool(contact),
                "active": bool(active),
                "center_distance": round(center_dist, 2),
                "speed_a": round(speed_a, 2),
                "speed_b": round(speed_b, 2),
                "a_bbox": a["bbox"],
                "b_bbox": b["bbox"],
                "contacts": contacts
            })

    events = []

    for pair_key, sigs in pair_signals.items():
        sigs = sorted(sigs, key=lambda x: x["time_sec"])
        start = 0

        while start < len(sigs):
            start_time = sigs[start]["time_sec"]
            window = []

            for j in range(start, len(sigs)):
                if sigs[j]["time_sec"] - start_time <= FIGHT_WINDOW_SEC:
                    window.append(sigs[j])
                else:
                    break

            if len(window) >= 2:
                signal_frames = [x for x in window if x["signal"]]

                # 사용자가 말한 기준: 2회 이상 반복
                if len(signal_frames) >= FIGHT_MIN_CONTACT_COUNT:
                    ev = {
                        "event_class": "fight",
                        "detected": True,
                        "track_pair": list(pair_key),
                        "start_frame": int(window[0]["frame_id"]),
                        "end_frame": int(window[-1]["frame_id"]),
                        "start_time": round(window[0]["time_sec"], 3),
                        "end_time": round(window[-1]["time_sec"], 3),
                        "confidence": round(min(0.95, 0.50 + 0.12 * len(signal_frames)), 3),
                        "reason": "손/발 후보 영역이 상대 bbox와 겹치거나 가까워지는 상황이 짧은 구간에서 2회 이상 반복됨.",
                        "evidence": {
                            "contact_count": len(signal_frames),
                            "window_count": len(window),
                            "sample_frames": signal_frames[:5]
                        }
                    }
                    events.append(ev)
                    start += max(1, len(window))
                else:
                    start += 1
            else:
                start += 1

    return merge_events(events, "fight", FIGHT_MERGE_GAP_SEC)


In [12]:
# ==============================
# Cell 12. 스토킹 탐지
# ==============================
def is_follower_behind_target(follower_point, target_point, target_dir):
    # target의 이동방향 벡터 기준으로 follower가 뒤쪽에 있는지 검사.
    # target_dir dot (follower - target) < 0 이면 뒤쪽.
    if norm(target_dir) <= 1e-6:
        return False, 0.0

    rel = vec(target_point["foot_point"], follower_point["foot_point"])
    dot = target_dir[0] * rel[0] + target_dir[1] * rel[1]

    behind = dot <= -STALK_BEHIND_DOT_TH
    return behind, round(dot, 3)

def detect_stalking(tracks, frame_map):
    if len(tracks) < 2:
        return []

    tmaps = {tid: track_point_map(pts) for tid, pts in tracks.items()}
    directional_signals = defaultdict(list)

    for frame_id in sorted(frame_map.keys()):
        persons = frame_map[frame_id]
        if len(persons) < 2:
            continue

        tids = sorted(persons.keys())

        for follower_id in tids:
            for target_id in tids:
                if follower_id == target_id:
                    continue

                follower = persons[follower_id]
                target = persons[target_id]

                follower_dir = compute_direction(tmaps[follower_id], frame_id)
                target_dir = compute_direction(tmaps[target_id], frame_id)

                sim = cosine(follower_dir, target_dir)
                distance = point_dist(follower["foot_point"], target["foot_point"])
                behind, behind_dot = is_follower_behind_target(follower, target, target_dir)

                follower_speed = compute_speed(tmaps[follower_id], frame_id)
                target_speed = compute_speed(tmaps[target_id], frame_id)

                distance_ok = STALK_MIN_DIST <= distance <= STALK_MAX_DIST
                direction_ok = sim >= STALK_DIR_SIM_TH
                moving_ok = follower_speed > 2 and target_speed > 2

                signal = distance_ok and direction_ok and moving_ok and behind

                pair_key = (str(follower_id), str(target_id))
                directional_signals[pair_key].append({
                    "frame_id": int(frame_id),
                    "time_sec": round(float(follower["time_sec"]), 3),
                    "signal": bool(signal),
                    "distance": round(distance, 2),
                    "direction_similarity": round(sim, 3),
                    "behind": bool(behind),
                    "behind_dot": behind_dot,
                    "follower_speed": round(follower_speed, 2),
                    "target_speed": round(target_speed, 2),
                    "follower_bbox": follower["bbox"],
                    "target_bbox": target["bbox"]
                })

    events = []

    for pair_key, sigs in directional_signals.items():
        sigs = sorted(sigs, key=lambda x: x["time_sec"])
        start = 0

        while start < len(sigs):
            start_time = sigs[start]["time_sec"]
            window = []

            for j in range(start, len(sigs)):
                if sigs[j]["time_sec"] - start_time <= STALK_WINDOW_SEC:
                    window.append(sigs[j])
                else:
                    break

            if len(window) >= 5:
                positives = [x for x in window if x["signal"]]
                ratio = len(positives) / len(window)
                dstd = std([x["distance"] for x in positives])

                if ratio >= STALK_MIN_RATIO and dstd <= STALK_DISTANCE_STD_TH:
                    follower_id, target_id = pair_key

                    ev = {
                        "event_class": "stalking",
                        "detected": True,
                        "stalking_type": "behind_following",
                        "follower_track_id": follower_id,
                        "target_track_id": target_id,
                        "start_frame": int(window[0]["frame_id"]),
                        "end_frame": int(window[-1]["frame_id"]),
                        "start_time": round(window[0]["time_sec"], 3),
                        "end_time": round(window[-1]["time_sec"], 3),
                        "confidence": round(min(0.93, 0.50 + ratio * 0.38), 3),
                        "reason": "한 사람이 다른 사람의 진행 방향 뒤쪽에서 일정 거리와 유사한 이동 방향을 유지함.",
                        "evidence": {
                            "signal_ratio": round(ratio, 3),
                            "signal_count": len(positives),
                            "window_count": len(window),
                            "distance_std": round(dstd, 3),
                            "sample_frames": positives[:5]
                        }
                    }
                    events.append(ev)
                    start += max(1, len(window))
                else:
                    start += 1
            else:
                start += 1

    return merge_events(events, "stalking", STALK_MERGE_GAP_SEC)


In [13]:
# ==============================
# Cell 13. 이벤트 프레임/클립 추출
# ==============================
def clip_time_to_frames(start_time, end_time, fps, frame_count=None):
    s = max(0, int((start_time - CLIP_MARGIN_SEC) * fps))
    e = max(s + 1, int((end_time + CLIP_MARGIN_SEC) * fps))

    if frame_count is not None and frame_count > 0:
        e = min(e, frame_count - 1)

    return s, e

def draw_event_boxes(frame, event, frame_points):
    img = frame.copy()

    if event["event_class"] == "fight":
        ids = set(map(str, event.get("track_pair", [])))
    elif event["event_class"] == "stalking":
        ids = {
            str(event.get("follower_track_id")),
            str(event.get("target_track_id"))
        }
    else:
        ids = set()

    for tid, p in frame_points.items():
        if str(tid) not in ids:
            continue

        x1, y1, x2, y2 = map(int, p["bbox"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(
            img,
            f"{event['event_class']} tid={tid}",
            (x1, max(0, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 255),
            2
        )

    return img

def extract_event_artifacts(video_path, video_name, events, frame_map, video_info):
    # 영상이 있으면 이벤트 시작 프레임 이미지와 이벤트 clip을 저장
    if video_path is None or not Path(video_path).exists() or not events:
        return []

    artifacts = []
    fps = video_info.get("fps") or DEFAULT_FPS
    frame_count = video_info.get("frame_count")

    for idx, ev in enumerate(events):
        event_id = f"{Path(str(video_name)).stem}_{idx:03d}_{ev['event_class']}"
        start_f, end_f = clip_time_to_frames(ev["start_time"], ev["end_time"], fps, frame_count)

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        frame_img_path = EVENT_FRAME_DIR / f"{event_id}_start.jpg"
        clip_path = EVENT_CLIP_DIR / f"{event_id}.mp4"

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
        ret, frame = cap.read()
        if ret:
            nearest_frame_id = min(frame_map.keys(), key=lambda x: abs(x - start_f)) if frame_map else start_f
            drawn = draw_event_boxes(frame, ev, frame_map.get(nearest_frame_id, {}))
            cv2.imwrite(str(frame_img_path), drawn)

        clip_saved = False

        if EXTRACT_EVENT_CLIPS:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            out = cv2.VideoWriter(str(clip_path), fourcc, fps, (width, height))

            cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
            cur = start_f

            while cur <= end_f:
                ret, frame = cap.read()
                if not ret:
                    break

                nearest_frame_id = min(frame_map.keys(), key=lambda x: abs(x - cur)) if frame_map else cur
                drawn = draw_event_boxes(frame, ev, frame_map.get(nearest_frame_id, {}))
                out.write(drawn)
                cur += 1

            out.release()
            clip_saved = clip_path.exists() and clip_path.stat().st_size > 0

        cap.release()

        artifact = {
            "event_index": idx,
            "event_class": ev["event_class"],
            "start_frame_image": str(frame_img_path) if frame_img_path.exists() else None,
            "clip_path": str(clip_path) if clip_saved else None
        }
        artifacts.append(artifact)

        ev["artifact"] = artifact

    return artifacts


In [14]:
# ==============================
# Cell 14. 파일 하나 처리 함수
# ==============================
def process_one_json(json_path):
    json_path = Path(json_path)
    raw = load_json(json_path)

    # 영상 매칭을 먼저 해서 fps 확보
    video_path = find_matching_video(json_path, raw)
    video_info = get_video_info(video_path)

    video_name, detections_by_frame = normalize_cf2_json(
        raw,
        json_path=json_path,
        fps=video_info.get("fps", DEFAULT_FPS)
    )

    tracks = build_tracks_from_detections(detections_by_frame)
    frame_map = build_frame_map(tracks)

    fight_events = detect_fight(tracks, frame_map)
    stalking_events = detect_stalking(tracks, frame_map)

    final_prediction = decide_final(fight_events, stalking_events)
    all_events = final_prediction.get("predictions", [])

    artifacts = []
    if video_path is not None and all_events:
        artifacts = extract_event_artifacts(
            video_path=video_path,
            video_name=video_name,
            events=all_events,
            frame_map=frame_map,
            video_info=video_info
        )

    output = {
        "video_name": str(video_name),
        "json_file": str(json_path),
        "matched_video": str(video_path) if video_path is not None else None,
        "task": "fight_and_stalking_detection_bd2",
        "input_summary": {
            "person_detection_frame_count": len(detections_by_frame),
            "person_detection_count": sum(len(v) for v in detections_by_frame.values()),
            "person_track_count": len(tracks),
            "video_info": video_info
        },
        "decision_rule": {
            "fight": [
                "person bbox에서 손/발 후보 bbox를 근사한다.",
                "손/발 후보 bbox가 상대 person bbox와 겹치거나 가까운 경우 contact 후보로 본다.",
                "3초 구간에서 contact+active 후보가 2회 이상 반복되면 fight 후보로 판정한다."
            ],
            "stalking": [
                "두 사람 track을 follower -> target 방향 관계로 검사한다.",
                "follower가 target의 진행 방향 뒤쪽에 있는지 확인한다.",
                "일정 거리 범위에서 같은 방향으로 이동하는 상태가 5초 구간에서 기준 비율 이상이면 stalking 후보로 판정한다."
            ]
        },
        "parameters": {
            "min_track_length": MIN_TRACK_LENGTH,
            "fight_window_sec": FIGHT_WINDOW_SEC,
            "fight_min_contact_count": FIGHT_MIN_CONTACT_COUNT,
            "fight_limb_near_threshold": FIGHT_LIMB_NEAR_TH,
            "stalking_window_sec": STALK_WINDOW_SEC,
            "stalking_min_ratio": STALK_MIN_RATIO,
            "stalking_min_distance": STALK_MIN_DIST,
            "stalking_max_distance": STALK_MAX_DIST,
            "stalking_direction_similarity_threshold": STALK_DIR_SIM_TH
        },
        "track_summaries": summarize_tracks(tracks),
        "fight_events": fight_events,
        "stalking_events": stalking_events,
        "event_artifacts": artifacts,
        "final_prediction": final_prediction
    }

    out_path = OUTPUT_DIR / f"{json_path.stem}_bd2_prediction.json"
    save_json(output, out_path)

    return output, out_path


In [15]:
# ==============================
# Cell 15. cf2 폴더 전체 실행
# ==============================
all_results = []
failed = []

if len(json_files) == 0:
    print("처리할 JSON이 없습니다. CF2_DIR 경로를 확인하세요:", CF2_DIR)
else:
    for jp in tqdm(json_files, desc="bd2 processing"):
        try:
            result, out_path = process_one_json(jp)
            all_results.append({
                "video_name": result["video_name"],
                "json_file": result["json_file"],
                "matched_video": result["matched_video"],
                "output_json": str(out_path),
                "person_track_count": result["input_summary"]["person_track_count"],
                "fight_count": len(result["fight_events"]),
                "stalking_count": len(result["stalking_events"]),
                "detected": result["final_prediction"]["detected"],
                "main_event": result["final_prediction"]["main_event"],
                "confidence": result["final_prediction"]["confidence"]
            })
        except Exception as e:
            failed.append({
                "json_file": str(jp),
                "error": repr(e)
            })

summary = {
    "task": "bd2_batch_summary",
    "cf2_dir": str(CF2_DIR),
    "sample_dir": str(SAMPLE_DIR),
    "output_dir": str(OUTPUT_DIR),
    "total_json": len(json_files),
    "success_count": len(all_results),
    "failed_count": len(failed),
    "detected_count": sum(1 for r in all_results if r["detected"]),
    "fight_video_count": sum(1 for r in all_results if r["fight_count"] > 0),
    "stalking_video_count": sum(1 for r in all_results if r["stalking_count"] > 0),
    "results": all_results,
    "failed": failed
}

summary_path = OUTPUT_DIR / "bd2_batch_summary.json"
save_json(summary, summary_path)

print("===================================")
print("bd2 전체 처리 완료")
print("성공:", len(all_results), "/", len(json_files))
print("실패:", len(failed))
print("탐지 영상 수:", summary["detected_count"])
print("싸움 후보 영상 수:", summary["fight_video_count"])
print("스토킹 후보 영상 수:", summary["stalking_video_count"])
print("summary 저장:", summary_path)


bd2 processing:   0%|          | 0/10 [00:00<?, ?it/s]

bd2 전체 처리 완료
성공: 10 / 10
실패: 0
탐지 영상 수: 4
싸움 후보 영상 수: 4
스토킹 후보 영상 수: 3
summary 저장: C:\Users\우성한\outputs\prediction\bd2\bd2_batch_summary.json


In [16]:
# ==============================
# Cell 16. 결과 상위 목록 확인
# ==============================
# confidence 높은 순서로 확인
top_results = sorted(all_results, key=lambda x: x["confidence"], reverse=True)

for r in top_results[:30]:
    print(
        f"[{r['main_event']}] conf={r['confidence']} "
        f"fight={r['fight_count']} stalking={r['stalking_count']} "
        f"tracks={r['person_track_count']} | {Path(r['json_file']).name}"
    )
    print(" ->", r["output_json"])


[fight] conf=0.95 fight=6 stalking=0 tracks=20 | es_e3768_c2.json
 -> C:\Users\우성한\outputs\prediction\bd2\es_e3768_c2_bd2_prediction.json
[fight] conf=0.95 fight=29 stalking=3 tracks=25 | md_e2122_c1.json
 -> C:\Users\우성한\outputs\prediction\bd2\md_e2122_c1_bd2_prediction.json
[fight] conf=0.95 fight=29 stalking=2 tracks=26 | md_e2122_c2.json
 -> C:\Users\우성한\outputs\prediction\bd2\md_e2122_c2_bd2_prediction.json
[fight] conf=0.95 fight=3 stalking=1 tracks=7 | mp_e2730_c2.json
 -> C:\Users\우성한\outputs\prediction\bd2\mp_e2730_c2_bd2_prediction.json
[none] conf=0.0 fight=0 stalking=0 tracks=5 | bs_e4387_c1.json
 -> C:\Users\우성한\outputs\prediction\bd2\bs_e4387_c1_bd2_prediction.json
[none] conf=0.0 fight=0 stalking=0 tracks=2 | bs_e4387_c2.json
 -> C:\Users\우성한\outputs\prediction\bd2\bs_e4387_c2_bd2_prediction.json
[none] conf=0.0 fight=0 stalking=0 tracks=5 | es_e3768_c1.json
 -> C:\Users\우성한\outputs\prediction\bd2\es_e3768_c1_bd2_prediction.json
[none] conf=0.0 fight=0 stalking=0 tracks=

In [17]:
# ==============================
# Cell 17. 실패 파일 확인
# ==============================
if failed:
    print("실패 파일:")
    for f in failed[:30]:
        print("-", f["json_file"])
        print(" ", f["error"])
else:
    print("실패 없음")


실패 없음


In [21]:
# ==============================
# bd2 결과에서 탐지 파일 포괄 출력
# ==============================
import json
from pathlib import Path

SUMMARY_JSON = Path(r"C:\Users\우성한\outputs\prediction\bd2\bd2_batch_summary.json")

with open(SUMMARY_JSON, "r", encoding="utf-8") as f:
    summary = json.load(f)

results = summary.get("results", [])

print("전체 result 수:", len(results))
print("탐지 영상 수:", summary.get("detected_count"))
print("싸움 후보 영상 수:", summary.get("fight_video_count"))
print("스토킹 후보 영상 수:", summary.get("stalking_video_count"))

print("\n================================")
print("탐지된 파일 목록")
print("================================")

for item in results:
    # 가능한 파일명 후보들
    json_file = (
        item.get("json_file")
        or item.get("input_json")
        or item.get("cf2_json")
        or item.get("json_name")
        or item.get("file_name")
        or item.get("video_name")
        or "unknown_json"
    )

    video_file = (
        item.get("video_file")
        or item.get("matched_video")
        or item.get("video_path")
        or "unknown_video"
    )

    # final_prediction 위치 후보
    final_prediction = (
        item.get("final_prediction")
        or item.get("prediction")
        or item.get("result", {}).get("final_prediction", {})
        or {}
    )

    detected = final_prediction.get("detected", item.get("detected", False))
    main_event = final_prediction.get("main_event", item.get("main_event", "none"))
    confidence = final_prediction.get("confidence", item.get("confidence", 0))

    predictions = final_prediction.get("predictions", item.get("predictions", []))

    # item 바로 아래에 fight_events/stalking_events가 있는 경우도 처리
    fight_events = item.get("fight_events", [])
    stalking_events = item.get("stalking_events", [])

    has_fight = (
        bool(fight_events)
        or any(p.get("event_class") == "fight" for p in predictions)
        or item.get("has_fight", False)
    )

    has_stalking = (
        bool(stalking_events)
        or any(p.get("event_class") == "stalking" for p in predictions)
        or item.get("has_stalking", False)
    )

    if detected or has_fight or has_stalking:
        labels = []
        if has_fight:
            labels.append("fight")
        if has_stalking:
            labels.append("stalking")
        if not labels and main_event != "none":
            labels.append(main_event)

        print(f"[{', '.join(labels)}] conf={confidence}")
        print(f"  json : {json_file}")
        print(f"  video: {video_file}")
        print()

전체 result 수: 10
탐지 영상 수: 4
싸움 후보 영상 수: 4
스토킹 후보 영상 수: 3

탐지된 파일 목록
[fight] conf=0.95
  json : C:\Users\우성한\cf2\es_e3768_c2.json
  video: unknown_video

[fight] conf=0.95
  json : C:\Users\우성한\cf2\md_e2122_c1.json
  video: unknown_video

[fight] conf=0.95
  json : C:\Users\우성한\cf2\md_e2122_c2.json
  video: unknown_video

[fight] conf=0.95
  json : C:\Users\우성한\cf2\mp_e2730_c2.json
  video: unknown_video

